What this is: the weather fetch, done with the same discipline as the tennis files. Each API response is saved exactly as returned (write-once, never overwritten), with the request and fetch time stored beside it, and a manifest with checksums is written at the end. The API has no commit number to pin, so the stored files are the source of truth. That matters for a reason specific to weather: recent reanalysis data gets revised slightly over time, so refetching later could return different numbers.

The order is: settings → load the list → test the model setting → pilot 3 editions → full run → validate and write the manifest.

# Authentication and settings

Switch on Notebook access for both secrets.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

# Load the edition list

In [ ]:
import pandas as pd
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location=REGION)
editions = bq.query(
    f"SELECT * FROM `{PROJECT_ID}.tennis_clean.tournament_editions` ORDER BY tourney_id",
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000),
).to_dataframe()
print(len(editions), "editions loaded")   # expect 873

# The fetch function

In [ ]:
import json, time, hashlib
import requests
from datetime import datetime, timezone
from google.cloud import storage

gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET_NAME)

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
MODEL = "era5"      # pinned: the default model changes over time
HOURLY = ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "surface_pressure",
          "precipitation", "wind_speed_10m", "wind_gusts_10m", "wind_direction_10m", "cloud_cover"]
RAW_PREFIX = f"raw/open_meteo/archive/model={MODEL}"

def fetch_edition(row, model=MODEL, store=True):
    """Fetch one edition's weather. With store=True: skip if already saved, else save write-once."""
    blob = bucket.blob(f"{RAW_PREFIX}/{row.tourney_id}.json")
    if store and blob.exists():
        return "already stored", None

    params = {
        "latitude": round(float(row.latitude), 4), "longitude": round(float(row.longitude), 4),
        "start_date": str(row.start_date), "end_date": str(row.end_date),
        "hourly": ",".join(HOURLY), "timezone": "auto",
    }
    if model:
        params["models"] = model

    for attempt in range(5):                       # retry on busy or temporary server errors
        r = requests.get(ARCHIVE_URL, params=params, timeout=60)
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(2 * 2 ** attempt)           # wait 2, 4, 8, 16, 32 seconds
            continue
        r.raise_for_status()
        break
    else:
        raise RuntimeError(f"Gave up on {row.tourney_id} (HTTP {r.status_code})")

    record = {
        "tourney_id": row.tourney_id,
        "request": {"url": ARCHIVE_URL, "params": params},
        "fetched_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "response": r.json(),                      # exactly as returned
    }
    if store:
        blob.upload_from_string(json.dumps(record), content_type="application/json",
                                if_generation_match=0)
    return "fetched", record

# Test that the model setting works (nothing is saved)

For a 2024 date, these should be different, because the default would use a finer, newer model. If they're identical either the setting isn't being applied or the default is ERA5 already, and we'd check the docs before fetching 873 files.

In [ ]:
row = next(editions[editions["start_date"].astype(str).str.startswith("2024-06")].itertuples())
_, pinned = fetch_edition(row, model="era5", store=False)
time.sleep(1)
_, default = fetch_edition(row, model=None, store=False)

a = pinned["response"]["hourly"]["temperature_2m"]
b = default["response"]["hourly"]["temperature_2m"]
print(row.place, row.start_date)
print("era5   first 6 hours:", a[:6])
print("default first 6 hours:", b[:6])
print("identical:", a == b)

The model pin works. The two series differ (identical: False), as expected for a 2024 date: unpinned, the API used a different, newer model. The gap is about 0.5 to 1 °C in these early-morning hours, small but systematic. Without the pin, the weather source would have changed around 2017 and the model could have learned that change as if it were a real effect.

# Pilot, with three real editions saved

Print so can check that the timezone matches the venue, the units look right (°C, km/h, mm), and the values are plausible (no long runs of None).

In [ ]:
pilot = pd.concat([
    editions[editions["tourney_level"] == "G"].iloc[[0]],
    editions[editions["tourney_level"] == "I"].iloc[[100]],
    editions[editions["tourney_level"] == "P"].iloc[[-1]],
])
for row in pilot.itertuples():
    status, _ = fetch_edition(row)
    print(row.tourney_id, "|", row.place, "|", status)
    time.sleep(0.5)

rec = json.loads(bucket.blob(f"{RAW_PREFIX}/{pilot.iloc[0].tourney_id}.json").download_as_text())
resp = rec["response"]
print({k: resp.get(k) for k in ["latitude", "longitude", "elevation", "timezone", "hourly_units"]})
pd.DataFrame(resp["hourly"]).head(6)

# The full run
Expect 10 to 20 minutes. It's safe to interrupt, and if Colab disconnects, run Cells 1 to 3 and then Cell 6 again. Anything already saved is skipped. If failed isn't empty, just run Cell 6 once more. Check Open-Meteo's terms and pricing page for the current free-tier limits. 873 small requests at this pace should be well within them, but they change over time.

In [ ]:
failed, counts = [], {"fetched": 0, "already stored": 0}
for i, row in enumerate(editions.itertuples(), 1):
    try:
        status, _ = fetch_edition(row)
        counts[status] += 1
        if status == "fetched":
            time.sleep(0.5)                        # about 2 requests a second at most
    except Exception as e:
        failed.append((row.tourney_id, repr(e)[:150]))
    if i % 100 == 0:
        print(i, counts, "failed:", len(failed))

print("Done:", counts, "| failed:", len(failed))
failed[:5]

# Validate what's stored

In [ ]:
entries = []
for b in gcs.list_blobs(BUCKET_NAME, prefix=f"{RAW_PREFIX}/"):
    data = b.download_as_bytes()
    rec = json.loads(data)
    hourly = rec["response"]["hourly"]
    entries.append({
        "file": b.name, "tourney_id": b.name.split("/")[-1][:-5], "bytes": len(data),
        "sha256": hashlib.sha256(data).hexdigest(), "n_hours": len(hourly["time"]),
        "null_values": sum(v is None for k, vals in hourly.items() if k != "time" for v in vals),
        "fetched_at_utc": rec["fetched_at_utc"],
    })

check = pd.DataFrame(entries).merge(editions[["tourney_id", "start_date", "end_date"]], on="tourney_id", how="left")
check["expected_hours"] = ((pd.to_datetime(check["end_date"]) - pd.to_datetime(check["start_date"])).dt.days + 1) * 24
check["hours_diff"] = check["n_hours"] - check["expected_hours"]

print("files:", len(check), "| editions with no file:", len(set(editions["tourney_id"]) - set(check["tourney_id"])))
print("files with hour count off by more than 2:", (check["hours_diff"].abs() > 2).sum())
print("total null values:", check["null_values"].sum())
print("total size (MB):", round(check["bytes"].sum() / 1e6, 1))

The data is complete. 873 files for 873 editions, none missing, hour counts match the requested windows, and zero null values across every variable.
It's small. 14 MB in total, so storage cost is negligible.
The values look plausible. About 9 to 11 °C in the early hours of a June day in Birmingham is reasonable.

# Save manifest

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
manifest_path = f"manifests/open_meteo_archive_{MODEL}_{stamp}.json"
bucket.blob(manifest_path).upload_from_string(
    json.dumps({"endpoint": ARCHIVE_URL, "model": MODEL, "hourly_variables": HOURLY,
                "files": entries}, indent=1),
    content_type="application/json")
print("Saved", manifest_path, "with", len(entries), "files")

# Check manifest exists

In [ ]:
for b in gcs.list_blobs(BUCKET_NAME, prefix="manifests/open_meteo_archive"):
    print(b.name)

README additions: the ERA5 resolution of about 25 km, that weather is city-level and the free tier is non-commercial and needs attribution to Open-Meteo, and that re-fetching later may return slightly revised values, which is why the stored files are the source of truth.